In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# read images
image_folder = "gravity_falls_dataset/puzzle_2x2"
image_files = sorted(os.listdir(image_folder))  # optional: sort for consistent IDs

images = []
for file in image_files:
    path = os.path.join(image_folder, file)
    img = cv2.imread(path)
    if img is not None:
        images.append(img)

In [ ]:
def show_two_images(img, diagnosed_img, i):
    plt.figure(figsize=(8,4))

    # First image
    plt.subplot(1, 2, 1)
    plt.imshow(img[..., ::-1])  # BGR → RGB if using OpenCV
    plt.title(f"Image {i+1}")
    plt.axis("off")

    # Second image
    plt.subplot(1, 2, 2)
    plt.imshow(diagnosed_img[..., ::-1])
    plt.title(f"Diagnosed Image {i+1}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
#output of step 1 will be stored in denoised images[]
denoised_images = []

In [ ]:
# bilateral filter denoising and display
for i, img in enumerate(images):
    if i == 5:
        break
    denoised = cv2.bilateralFilter(img, 5, 75, 75)
    denoised_images.append(denoised)
    show_two_images(img, denoised, i)

In [ ]:
# Median filter denoising and display
for i, img in enumerate(images):
    if i == 5:
        break
    denoised = cv2.medianBlur(img, 3)
    denoised_images.append(denoised)
    show_two_images(img, denoised, i)

In [ ]:
#gaussian filter
for i, img in enumerate(images):
    if i == 5:
        break
    denoised = cv2.GaussianBlur(img, (3, 3), 0)
    denoised_images.append(denoised)
    denoised_images.append(denoised)


step 2 Image Enhancement

In [ ]:
#converting from bgr to grey scale because some filters can use only grey scale 
lab_images = []

for i, denoised in enumerate(denoised_images):
    if i == 5:
        break
    #convert from bgr to lab 
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    lab_images.append((L, A, B))


Contrast:


In [ ]:
#option 1 in contrast enhancement(on grey scale) using the clahe algorithms L only because L is just the intenisty of the img
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_images = []

for i, (L, A, B) in enumerate(lab_images):
    L_clahe = clahe.apply(L)
    clahe_images.append((L_clahe, A, B))



In [ ]:
#option 2 contrast enhancement using Histogram Equalization on L Channel
he_images = []

for i, (L, A, B) in enumerate(lab_images):
    L_he = cv2.equalizeHist(L)
    he_images.append((L_he, A, B))


In [ ]:
#Gamma correction → adjusts overall brightness/midtone, may make edges more visible
def apply_gamma(L, gamma=1.2):
    # Normalize to [0,1], apply gamma, scale back to [0,255]
    L_float = L / 255.0
    L_gamma = np.power(L_float, 1.0 / gamma) * 255.0
    return L_gamma.astype(np.uint8)


In [ ]:
#apply gamma correction
  
L = apply_gamma(L, gamma=1.2)  # adjust gamma as needed

Edge enhacement:


In [ ]:
enhanced_images = []

In [ ]:

#option 1 in edge enhancement
for i, (L, A, B) in enumerate(clahe_images):
    if i == 5:
        break

    # --- Sharpen (Unsharp Mask) ---
    #edge extraction
    blur = cv2.GaussianBlur(L, (0, 0), sigmaX=1.2)
    detail = cv2.subtract(L, blur)
    L_sharp = cv2.add(L, detail)

    # Merge back to LAB
    enhanced_lab = cv2.merge([L_sharp, A, B])
    enhanced_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

    #store the imgs
    enhanced_images.append(enhanced_bgr)
    # Show input denoised vs enhanced
    show_two_images(denoised_images[i], enhanced_bgr, i)



In [ ]:
#option 2 in edge enhancement 
for i, (L, A, B) in enumerate(he_images):
    if i == 5:
        break

    # Laplacian sharpening
    lap = cv2.Laplacian(L, cv2.CV_64F)
    L_sharp = cv2.add(L, cv2.convertScaleAbs(lap))

    # Merge back LAB
    enhanced_lab = cv2.merge([L_sharp, A, B])
    enhanced_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)
     #store the imgs
    enhanced_images.append(enhanced_bgr)
    # Show original denoised vs enhanced
    show_two_images(denoised_images[i], enhanced_bgr, i)


FUNCTIONS TO EVALUATE:

In [ ]:
def rms_contrast(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.std(gray)
#High RMS → image has stronger brightness variation → edges and puzzle pieces stand out
#Low RMS → image looks flat

In [ ]:
def edge_strength(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    return np.sum(edges > 0)

#More edge pixels → edges are clearer and better for contour extraction
#Too many edges/noisy → sharpening may be too strong


In [ ]:
#to know how i should adjust the gamma in gamma correction 
def compute_mean_intensity(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)
#Check if image is too dark or bright (ideal ~120–150)

In [ ]:
print(f"{'Index':<5} {'RMS':<10} {'EdgePixels':<12} {'MeanIntensity':<12}")
for i, img in enumerate(enhanced_images):
    rms = rms_contrast(img)
    edges = edge_strength(img)
    mean_int = compute_mean_intensity(img)

    
    print(f"{i:<5} {rms:<10.2f} {edges:<12} {mean_int:<12.2f}")